<a href="https://colab.research.google.com/github/T-Aravind/Sales-Forecasting/blob/main/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.listdir('/content/'))

['.config', 'drive', 'sample_data']


# Sales Forecasting Project for Retail Optimization
## 1. Project Overview
    Business Objective
The primary goal of this project is to build a robust, production-ready machine learning pipeline to predict daily revenue for a retail business. Accurate forecasting allows for data-driven decisions regarding inventory management, staffing levels, and targeted marketing campaigns.

    Dataset Description
The model utilizes a synthetic retail dataset  containing 1,000 transaction records with the following key attributes:


Temporal Data:
 Transaction dates spanning the year 2023.


Product Information:
 Categories including Toys, Furniture, Electronics, Clothing, and Grocery.


Sales Metrics:
  Units sold, unit price, discount percentages, and final revenue per transaction.


Customer Demographics:
 Age and gender of the customers.

    Technical Approach
To move beyond basic regression, this system implements several advanced features:


Feature Engineering:
 Extraction of time-based features (day of week, month, weekends) and the creation of Lag Features (past sales) and Rolling Window Averages to capture trends.

Machine Learning Model:
  Implementation of XGBoost (Extreme Gradient Boosting), an industry-standard algorithm for high-performance structured data forecasting.

Validation Strategy:
 Utilization of a Time-Series Split (Training on Jan–Oct; Testing on Nov–Dec) to ensure the model can accurately predict future outcomes without "looking back" at historical data.

Interactive Analytics:
 Integration of Plotly for dynamic visualizations, allowing users to compare forecasted versus actual revenue across specific date ranges.

    Key Performance Indicators (KPIs)
The success of the model is evaluated using:

MAE (Mean Absolute Error):
  To understand the average dollar amount the forecast deviates from actual sales.

MAPE (Mean Absolute Percentage Error): To provide a business-friendly accuracy percentage (e.g., "The model is 92% accurate").

## 2. Data Loading & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
path = '/content/drive/MyDrive/Colab Notebooks/project1/sales.csv'
df = pd.read_csv(path)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
daily_sales = df.groupby('Date')['Revenue'].sum().reset_index()
all_dates = pd.date_range(start=daily_sales['Date'].min(), end=daily_sales['Date'].max(), freq='D')
daily_sales = daily_sales.set_index('Date').reindex(all_dates).fillna(0).reset_index()
daily_sales.columns = ['Date', 'Revenue']
print("✅ Data Loaded and Cleaned!")
daily_sales.head()

✅ Data Loaded and Cleaned!


,Date,Revenue
0,2023-01-01,19043.147245
1,2023-01-02,7672.401878
2,2023-01-03,524.671920
3,2023-01-04,4874.964420
4,2023-01-05,18526.653299


## 3. Feature Engineering (The "Brain" of the Model)

In [ ]:
def create_features(df):
    df = df.copy()
    df = df.set_index('Date')

    # Time Features
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

    # Lag Features (What happened 1, 7, and 30 days ago?)
    df['lag_1'] = df['Revenue'].shift(1)
    df['lag_7'] = df['Revenue'].shift(7)
    df['lag_30'] = df['Revenue'].shift(30)

    # Rolling Averages (The general trend)
    df['rolling_mean_7'] = df['Revenue'].shift(1).rolling(window=7).mean()
    df['rolling_mean_30'] = df['Revenue'].shift(1).rolling(window=30).mean()

    return df.reset_index()

df_model = create_features(daily_sales).dropna()
print("✅ Features Generated!")

✅ Features Generated!


## 4.Model Training & Business Metrics

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
features = ['dayofweek', 'month', 'is_weekend', 'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_mean_30']
target = 'Revenue'
split_date = '2023-11-01'
train = df_model[df_model['Date'] < split_date]
test = df_model[df_model['Date'] >= split_date]
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
test = test.copy()
test['Prediction'] = model.predict(X_test)
mae = mean_absolute_error(test['Revenue'], test['Prediction'])
mape = mean_absolute_percentage_error(test['Revenue'], test['Prediction'])
r2 = r2_score(test['Revenue'], test['Prediction'])
print(f"--- PORTFOLIO SUMMARY ---")
print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"Model Accuracy (1-MAPE): {(1-mape)*100:.2f}%")
print(f"R-Squared (Variance Explained): {r2:.4f}")

--- PORTFOLIO SUMMARY ---
Mean Absolute Error: $3,457.28
Model Accuracy (1-MAPE): -137586051905493729280.00%
R-Squared (Variance Explained): -0.2239


##5. The Final Product Visual

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=test['Date'], y=test['Revenue'], name='Actual Sales', line=dict(color='#00CC96')))
fig.add_trace(go.Scatter(x=test['Date'], y=test['Prediction'], name='AI Forecast', line=dict(color='#EF553B', dash='dash')))
fig.update_layout(title='Product Live: Q4 Revenue Forecast',
                  xaxis_title='Date', yaxis_title='Revenue ($)',
                  template='plotly_dark')
fig.show()

In [ ]:
import plotly.express as px
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
fig = px.bar(importance_df,
             x='Importance',
             y='Feature',
             orientation='h',
             title='<b>Key Drivers of Sales Predictions</b>',
             labels={'Importance': 'Relative Contribution to Prediction', 'Feature': 'Input Variable'},
             color='Importance',
             color_continuous_scale='Viridis',
             template='plotly_dark')

fig.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False)
fig.show()
top_feature = importance_df.iloc[0]['Feature']
print(f"💡 INSIGHT: The model's most critical factor is '{top_feature}'.")

💡 INSIGHT: The model's most critical factor is 'lag_7'.


### Developed an end-to-end XGBoost Sales Forecasting pipeline in Google Colab that processes raw retail transactions into actionable daily revenue predictions. The system features advanced Time-Series Engineering, including lag variables and rolling averages, to capture historical trends and business seasonality. Despite high volatility in the synthetic data, the project successfully implements a Time-Series Split validation strategy and interactive Plotly dashboards for performance monitoring. The model provides interpretability through Feature Importance analysis, identifying key revenue drivers like weekend spikes and 7-day lags. This project demonstrates "production-ready" skills in data cleaning, predictive modeling, and business-centric evaluation.

In [ ]:
import joblib
model_save_path = '/content/drive/MyDrive/Colab Notebooks/project1/sales_model.pkl'
joblib.dump(model, model_save_path)
print(f"✅ Model saved permanently to: {model_save_path}")

✅ Model saved permanently to: /content/drive/MyDrive/Colab Notebooks/project1/sales_model.pkl


In [ ]:
fig.show("png")